
# 00 — Data Audit and Research Design

Notebook ini merupakan **gerbang validasi** sebelum eksplorasi dan pemodelan.

Notebook dinyatakan lolos hanya jika:

1. workbook gabungan dan seluruh file sumber dapat dibaca;
2. nama kolom telah distandardisasi dan seluruh kolom wajib tersedia;
3. kunci utama pada dua analytical datasets bersifat unik;
4. agregasi file sumber sama dengan nilai yang ditanamkan pada dataset gabungan untuk seluruh kunci yang dapat dibandingkan;
5. *missing values* dan *unmatched keys* terdokumentasi;
6. target, exposure, prediktor, dan kelompok validasi telah ditetapkan secara eksplisit.

> **Batas interpretasi:** `graduate_output_intensity` adalah rasio arus lulusan terhadap stok mahasiswa terdaftar, bukan *graduation rate* atau *completion probability* cohort.


In [ ]:

from __future__ import annotations

import json
import platform
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Platform:", platform.platform())


In [ ]:

# Menentukan root proyek secara robust, baik notebook dijalankan dari folder proyek
# maupun langsung dari folder notebooks.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

COMBINED_FILE = PROCESSED_DIR / "Dataset_Gabungan_Kemdiktisaintek_2025.xlsx"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

assert COMBINED_FILE.exists(), f"Workbook gabungan tidak ditemukan: {COMBINED_FILE}"
assert RAW_DIR.exists(), f"Folder sumber asli tidak ditemukan: {RAW_DIR}"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("COMBINED_FILE:", COMBINED_FILE)
print("RAW_DIR      :", RAW_DIR)


In [ ]:

# Memuat fungsi validasi reusable dari folder src.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_validation import (
    add_keys,
    assert_unique_keys,
    collapse_repeated_metrics,
    normalize_category,
    normalize_text_key,
    read_excel_flexible,
    reconcile_metrics,
    require_columns,
    safe_divide,
    sha256_file,
    standardize_columns,
)



## 1. Membaca workbook gabungan dan memvalidasi sheet

Sheet yang diwajibkan:

- `Sumber_Data`
- `AD_Gender_Jenjang`
- `AD_STEM_Jenjang`
- `Audit_Gabung`
- `Audit_Unmatched`
- `Kamus_Variabel`


In [ ]:

required_sheets = {
    "Sumber_Data",
    "AD_Gender_Jenjang",
    "AD_STEM_Jenjang",
    "Audit_Gabung",
    "Audit_Unmatched",
    "Kamus_Variabel",
}

excel_file = pd.ExcelFile(COMBINED_FILE, engine="openpyxl")
available_sheets = set(excel_file.sheet_names)
missing_sheets = required_sheets - available_sheets

assert not missing_sheets, f"Sheet wajib tidak ditemukan: {sorted(missing_sheets)}"

source_meta = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="Sumber_Data", engine="openpyxl")
)
gender_df = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="AD_Gender_Jenjang", engine="openpyxl")
)
stem_df = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="AD_STEM_Jenjang", engine="openpyxl")
)
audit_merge = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="Audit_Gabung", engine="openpyxl")
)
audit_unmatched = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="Audit_Unmatched", engine="openpyxl")
)
data_dictionary = standardize_columns(
    pd.read_excel(COMBINED_FILE, sheet_name="Kamus_Variabel", engine="openpyxl")
)

print("Sheet tersedia:", excel_file.sheet_names)
print("AD_Gender_Jenjang:", gender_df.shape)
print("AD_STEM_Jenjang  :", stem_df.shape)



## 2. Konsistensi nama kolom dan perhitungan ulang indikator turunan

Formula Excel tidak dijadikan satu-satunya sumber indikator turunan. Semua rasio dihitung ulang dari kolom komponennya agar hasil tidak bergantung pada *cached formula values* workbook.


In [ ]:

EXPECTED_GENDER_COLUMNS = {
    "provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik",
    "jumlah_lulusan", "jumlah_terdaftar", "jumlah_aktif_a", "jumlah_mbkm_m",
    "jumlah_pt", "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan", "dosen_profesor", "dosen_senior",
    "dosen_total_sertifikasi", "dosen_tersertifikasi",
    "flag_match_terdaftar", "flag_match_aktif", "flag_match_pt", "flag_match_dosen",
}

EXPECTED_STEM_COLUMNS = {
    "provinsi_pt", "kabupaten_pt", "a_stem", "nm_jenj_didik",
    "jumlah_lulusan", "jumlah_mahasiswa_baru",
    "mhs_baru_pt_akreditasi_tinggi",
    "mhs_baru_prodi_akreditasi_tinggi",
    "mhs_baru_pt_prodi_tinggi",
    "jumlah_pt", "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan", "dosen_profesor", "dosen_senior",
    "dosen_total_sertifikasi", "dosen_tersertifikasi",
    "dosen_doktor_stem", "dosen_doktor_non_stem", "dosen_doktor_total",
    "dosen_doktor_sesuai_bidang",
    "flag_match_mahasiswa_baru", "flag_match_pt",
    "flag_match_dosen_doktor", "flag_match_dosen",
}

require_columns(gender_df, sorted(EXPECTED_GENDER_COLUMNS), "AD_Gender_Jenjang")
require_columns(stem_df, sorted(EXPECTED_STEM_COLUMNS), "AD_STEM_Jenjang")

# Seluruh variabel count dipaksa numerik.
gender_count_cols = [
    "jumlah_lulusan", "jumlah_terdaftar", "jumlah_aktif_a", "jumlah_mbkm_m",
    "jumlah_pt", "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan", "dosen_profesor", "dosen_senior",
    "dosen_total_sertifikasi", "dosen_tersertifikasi",
    "dosen_jk_total_jabatan", "dosen_jk_profesor",
    "dosen_jk_total_sertifikasi", "dosen_jk_tersertifikasi",
]
stem_count_cols = [
    "jumlah_lulusan", "jumlah_mahasiswa_baru",
    "mhs_baru_pt_akreditasi_tinggi",
    "mhs_baru_prodi_akreditasi_tinggi",
    "mhs_baru_pt_prodi_tinggi",
    "jumlah_pt", "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan", "dosen_profesor", "dosen_senior",
    "dosen_total_sertifikasi", "dosen_tersertifikasi",
    "dosen_doktor_stem", "dosen_doktor_non_stem",
    "dosen_doktor_total", "dosen_doktor_sesuai_bidang",
]

for col in gender_count_cols:
    if col in gender_df.columns:
        gender_df[col] = pd.to_numeric(gender_df[col], errors="coerce")

for col in stem_count_cols:
    if col in stem_df.columns:
        stem_df[col] = pd.to_numeric(stem_df[col], errors="coerce")

# Perhitungan ulang indikator Dataset A.
gender_df["jumlah_aktif_total"] = gender_df[
    ["jumlah_aktif_a", "jumlah_mbkm_m"]
].sum(axis=1, min_count=1)
gender_df["graduate_output_intensity"] = safe_divide(
    gender_df["jumlah_lulusan"], gender_df["jumlah_terdaftar"]
)
gender_df["lulusan_per_1000_terdaftar"] = (
    gender_df["graduate_output_intensity"] * 1000
)
gender_df["proporsi_mbkm"] = safe_divide(
    gender_df["jumlah_mbkm_m"], gender_df["jumlah_aktif_total"]
)
gender_df["proporsi_pt_akreditasi_tinggi"] = safe_divide(
    gender_df["jumlah_pt_akreditasi_tinggi"], gender_df["jumlah_pt"]
)
gender_df["proporsi_profesor"] = safe_divide(
    gender_df["dosen_profesor"], gender_df["dosen_total_jabatan"]
)
gender_df["proporsi_dosen_senior"] = safe_divide(
    gender_df["dosen_senior"], gender_df["dosen_total_jabatan"]
)
gender_df["proporsi_dosen_tersertifikasi"] = safe_divide(
    gender_df["dosen_tersertifikasi"], gender_df["dosen_total_sertifikasi"]
)

if {"dosen_jk_profesor", "dosen_jk_total_jabatan"}.issubset(gender_df.columns):
    gender_df["proporsi_profesor_jk"] = safe_divide(
        gender_df["dosen_jk_profesor"], gender_df["dosen_jk_total_jabatan"]
    )
if {"dosen_jk_tersertifikasi", "dosen_jk_total_sertifikasi"}.issubset(gender_df.columns):
    gender_df["proporsi_sertifikasi_jk"] = safe_divide(
        gender_df["dosen_jk_tersertifikasi"], gender_df["dosen_jk_total_sertifikasi"]
    )

# Perhitungan ulang indikator Dataset B.
stem_df["graduate_to_entry_flow_ratio"] = safe_divide(
    stem_df["jumlah_lulusan"], stem_df["jumlah_mahasiswa_baru"]
)
stem_df["lulusan_per_100_mahasiswa_baru"] = (
    stem_df["graduate_to_entry_flow_ratio"] * 100
)
stem_df["proporsi_mhs_baru_pt_akreditasi_tinggi"] = safe_divide(
    stem_df["mhs_baru_pt_akreditasi_tinggi"],
    stem_df["jumlah_mahasiswa_baru"],
)
stem_df["proporsi_mhs_baru_prodi_akreditasi_tinggi"] = safe_divide(
    stem_df["mhs_baru_prodi_akreditasi_tinggi"],
    stem_df["jumlah_mahasiswa_baru"],
)
stem_df["proporsi_mhs_baru_pt_prodi_tinggi"] = safe_divide(
    stem_df["mhs_baru_pt_prodi_tinggi"],
    stem_df["jumlah_mahasiswa_baru"],
)
stem_df["proporsi_pt_akreditasi_tinggi"] = safe_divide(
    stem_df["jumlah_pt_akreditasi_tinggi"], stem_df["jumlah_pt"]
)
stem_df["proporsi_profesor"] = safe_divide(
    stem_df["dosen_profesor"], stem_df["dosen_total_jabatan"]
)
stem_df["proporsi_dosen_senior"] = safe_divide(
    stem_df["dosen_senior"], stem_df["dosen_total_jabatan"]
)
stem_df["proporsi_dosen_tersertifikasi"] = safe_divide(
    stem_df["dosen_tersertifikasi"], stem_df["dosen_total_sertifikasi"]
)
stem_df["proporsi_doktor_stem"] = safe_divide(
    stem_df["dosen_doktor_stem"], stem_df["dosen_doktor_total"]
)

print("Kolom wajib tersedia dan indikator turunan berhasil dihitung ulang.")



## 3. Uji kunci unik

- Dataset A: `provinsi_pt × kabupaten_pt × jk × nm_jenj_didik`
- Dataset B: `provinsi_pt × kabupaten_pt × a_stem × nm_jenj_didik`

Tidak boleh ada dua baris dengan kunci yang sama.


In [ ]:

gender_df = add_keys(
    gender_df,
    include_gender=True,
    include_level=True,
)
stem_df = add_keys(
    stem_df,
    include_stem=True,
    include_level=True,
)

KEY_A = ["_prov_key", "_kab_key", "_jk_key", "_jenjang_key"]
KEY_B = ["_prov_key", "_kab_key", "_stem_key", "_jenjang_key"]
KEY_GEO = ["_prov_key", "_kab_key"]

duplicate_a = gender_df.loc[
    gender_df.duplicated(KEY_A, keep=False)
].sort_values(KEY_A)
duplicate_b = stem_df.loc[
    stem_df.duplicated(KEY_B, keep=False)
].sort_values(KEY_B)

pd.concat(
    [
        duplicate_a.assign(dataset="AD_Gender_Jenjang"),
        duplicate_b.assign(dataset="AD_STEM_Jenjang"),
    ],
    ignore_index=True,
).to_csv(OUTPUT_TABLE_DIR / "00_duplicate_keys.csv", index=False)

assert_unique_keys(gender_df, KEY_A, "AD_Gender_Jenjang")
assert_unique_keys(stem_df, KEY_B, "AD_STEM_Jenjang")

print(f"PASS — Dataset A: {len(gender_df):,} kunci unik.")
print(f"PASS — Dataset B: {len(stem_df):,} kunci unik.")



## 4. Dokumentasi missing values dan unmatched keys

Nilai kosong tidak otomatis diubah menjadi nol. Notebook menghasilkan:

- ringkasan *missing values* per kolom;
- distribusi flag hasil penggabungan;
- salinan tabel `Audit_Unmatched`;
- daftar baris dengan sedikitnya satu sumber yang tidak cocok.


In [ ]:

def missing_summary(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    result = pd.DataFrame(
        {
            "dataset": dataset_name,
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "missing_n": df.isna().sum().to_numpy(),
        }
    )
    result["missing_pct"] = result["missing_n"] / len(df) * 100
    return result.sort_values(
        ["missing_pct", "column"], ascending=[False, True]
    ).reset_index(drop=True)


missing_report = pd.concat(
    [
        missing_summary(gender_df, "AD_Gender_Jenjang"),
        missing_summary(stem_df, "AD_STEM_Jenjang"),
    ],
    ignore_index=True,
)
missing_report.to_csv(
    OUTPUT_TABLE_DIR / "00_missing_values.csv",
    index=False,
)

flag_cols_a = [c for c in gender_df.columns if c.startswith("flag_match_")]
flag_cols_b = [c for c in stem_df.columns if c.startswith("flag_match_")]

flag_report = []
for dataset_name, frame, columns in [
    ("AD_Gender_Jenjang", gender_df, flag_cols_a),
    ("AD_STEM_Jenjang", stem_df, flag_cols_b),
]:
    for col in columns:
        counts = frame[col].fillna("MISSING_FLAG").value_counts(dropna=False)
        for status, count in counts.items():
            flag_report.append(
                {
                    "dataset": dataset_name,
                    "flag_column": col,
                    "status": status,
                    "n": int(count),
                    "pct": float(count / len(frame) * 100),
                }
            )

flag_report = pd.DataFrame(flag_report)
flag_report.to_csv(
    OUTPUT_TABLE_DIR / "00_match_flag_summary.csv",
    index=False,
)

audit_unmatched = audit_unmatched.dropna(how="all").copy()
audit_unmatched.to_csv(
    OUTPUT_TABLE_DIR / "00_unmatched_keys.csv",
    index=False,
)

display(missing_report.head(20))
display(flag_report)
print("Jumlah baris Audit_Unmatched:", len(audit_unmatched))



## 5. Validasi file sumber dan rekonsiliasi agregasi

Rekonsiliasi dilakukan pada **kunci yang dapat dibandingkan**. Dataset gabungan dibangun dengan outcome lulusan sebagai tabel utama, sehingga sebagian kunci yang hanya terdapat pada data input dapat tetap menjadi `source_only`. Kondisi tersebut tidak dianggap kesalahan selama:

1. nilai pada seluruh kunci yang cocok identik;
2. jumlah `source_only` terdokumentasi;
3. tidak ada penggandaan nilai akibat *many-to-many join*.


In [ ]:

SOURCE_FILES = {
    "lulus": "Mahasiswa Lulus (Tanpa Akreditasi).xlsx",
    "terdaftar": "Mahasiswa Terdaftar.xlsx",
    "aktif": "Mahasiswa Aktif.xlsx",
    "mahasiswa_baru": "Data Mahasiswa Baru Tahun 2025.xlsx",
    "pt": "Perguruan Tinggi Terakreditasi.xlsx",
    "dosen_jabatan": "data set jumlah dosen berdasarkan jabatan akademik.xls",
    "dosen_sertifikasi": "data set jumlah dosen berdasarkan status sertifikasi.xls",
    "dosen_doktor": "data set jumlah dosen doktor bidang stem dan non stem.xls",
}

source_meta_used = source_meta.loc[
    source_meta["status_penggunaan"].eq("Digunakan")
].dropna(subset=["nama_file_asli"]).copy()

file_audit_rows = []
for _, row in source_meta_used.iterrows():
    path = RAW_DIR / row["nama_file_asli"]
    exists = path.exists()
    actual_sha = sha256_file(path) if exists else None
    expected_sha = str(row["sha256"]).strip().lower()
    sha_match = exists and actual_sha == expected_sha

    file_audit_rows.append(
        {
            "nama_file_asli": row["nama_file_asli"],
            "exists": exists,
            "expected_sha256": expected_sha,
            "actual_sha256": actual_sha,
            "sha256_match": sha_match,
            "expected_rows": row["jumlah_baris_data"],
        }
    )

file_audit = pd.DataFrame(file_audit_rows)
file_audit.to_csv(
    OUTPUT_TABLE_DIR / "00_source_file_integrity.csv",
    index=False,
)

assert file_audit["exists"].all(), (
    "Ada file sumber yang tidak ditemukan:\n"
    + file_audit.loc[~file_audit["exists"], "nama_file_asli"].to_string(index=False)
)
assert file_audit["sha256_match"].all(), (
    "Checksum SHA-256 file sumber berubah:\n"
    + file_audit.loc[
        ~file_audit["sha256_match"],
        ["nama_file_asli", "expected_sha256", "actual_sha256"],
    ].to_string(index=False)
)

display(file_audit[["nama_file_asli", "exists", "sha256_match", "expected_rows"]])
print("PASS — seluruh file sumber asli tersedia dan checksum sesuai.")


In [ ]:

# Memuat seluruh sumber asli menggunakan deteksi header fleksibel.
raw_lulus = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["lulus"],
    ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik", "a_stem", "jumlah_lulusan"],
)
raw_terdaftar = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["terdaftar"],
    ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik", "jumlah"],
)
raw_aktif = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["aktif"],
    ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik", "id_stat_mhs", "jumlah"],
)
raw_mahasiswa_baru = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["mahasiswa_baru"],
    [
        "provinsi_pt", "kabupaten_pt", "akred_pt", "nm_jenj_didik",
        "a_stem", "akred_prodi", "jumlah_mhs_baru",
    ],
)
raw_pt = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["pt"],
    ["provinsi_pt", "kabupaten_pt", "npsn", "akred_pt"],
)
raw_dosen_jabatan = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["dosen_jabatan"],
    ["provinsi_pt", "kabupaten_pt", "jabatan_akademik", "jumlah"],
)
raw_dosen_sertifikasi = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["dosen_sertifikasi"],
    ["provinsi_pt", "kabupaten_pt", "status_sertifikasi", "jumlah"],
)
raw_dosen_doktor = read_excel_flexible(
    RAW_DIR / SOURCE_FILES["dosen_doktor"],
    ["provinsi_pt", "kabupaten_pt", "jenjang_tertinggi", "bidang", "jumlah"],
)

raw_frames = {
    SOURCE_FILES["lulus"]: raw_lulus,
    SOURCE_FILES["terdaftar"]: raw_terdaftar,
    SOURCE_FILES["aktif"]: raw_aktif,
    SOURCE_FILES["mahasiswa_baru"]: raw_mahasiswa_baru,
    SOURCE_FILES["pt"]: raw_pt,
    SOURCE_FILES["dosen_jabatan"]: raw_dosen_jabatan,
    SOURCE_FILES["dosen_sertifikasi"]: raw_dosen_sertifikasi,
    SOURCE_FILES["dosen_doktor"]: raw_dosen_doktor,
}

row_count_checks = []
for filename, frame in raw_frames.items():
    expected = int(
        source_meta_used.loc[
            source_meta_used["nama_file_asli"].eq(filename),
            "jumlah_baris_data",
        ].iloc[0]
    )
    actual = len(frame)
    row_count_checks.append(
        {
            "nama_file_asli": filename,
            "expected_rows": expected,
            "actual_rows": actual,
            "status": "PASS" if expected == actual else "FAIL",
        }
    )

row_count_checks = pd.DataFrame(row_count_checks)
row_count_checks.to_csv(
    OUTPUT_TABLE_DIR / "00_source_row_count_check.csv",
    index=False,
)
display(row_count_checks)

assert row_count_checks["status"].eq("PASS").all(), (
    "Jumlah baris file sumber berbeda dari metadata:\n"
    + row_count_checks.loc[
        row_count_checks["status"].eq("FAIL")
    ].to_string(index=False)
)


In [ ]:

# ---------- Agregasi sumber: mahasiswa lulus ----------
raw_lulus["jumlah_lulusan"] = pd.to_numeric(
    raw_lulus["jumlah_lulusan"], errors="coerce"
)
raw_lulus = add_keys(
    raw_lulus,
    include_gender=True,
    include_level=True,
    include_stem=True,
)

lulus_a = (
    raw_lulus.loc[
        raw_lulus["jumlah_lulusan"].notna()
        & raw_lulus["_jk_key"].isin(["L", "P"])
    ]
    .groupby(KEY_A, dropna=False, as_index=False)["jumlah_lulusan"]
    .sum()
)

lulus_b = (
    raw_lulus.loc[raw_lulus["jumlah_lulusan"].notna()]
    .groupby(KEY_B, dropna=False, as_index=False)["jumlah_lulusan"]
    .sum()
)

# ---------- Agregasi sumber: mahasiswa terdaftar ----------
raw_terdaftar["jumlah"] = pd.to_numeric(raw_terdaftar["jumlah"], errors="coerce")
raw_terdaftar = add_keys(
    raw_terdaftar,
    include_gender=True,
    include_level=True,
)
terdaftar_a = (
    raw_terdaftar.loc[
        raw_terdaftar["jumlah"].notna()
        & raw_terdaftar["_jk_key"].isin(["L", "P"])
    ]
    .groupby(KEY_A, dropna=False, as_index=False)["jumlah"]
    .sum()
    .rename(columns={"jumlah": "jumlah_terdaftar"})
)

# ---------- Agregasi sumber: mahasiswa aktif A/M ----------
raw_aktif["jumlah"] = pd.to_numeric(raw_aktif["jumlah"], errors="coerce")
raw_aktif = add_keys(
    raw_aktif,
    include_gender=True,
    include_level=True,
)
raw_aktif["_status_key"] = raw_aktif["id_stat_mhs"].map(normalize_category)

active_long = (
    raw_aktif.loc[
        raw_aktif["jumlah"].notna()
        & raw_aktif["_jk_key"].isin(["L", "P"])
        & raw_aktif["_status_key"].isin(["A", "M"])
    ]
    .groupby(KEY_A + ["_status_key"], dropna=False, as_index=False)["jumlah"]
    .sum()
)
active_a = (
    active_long.pivot_table(
        index=KEY_A,
        columns="_status_key",
        values="jumlah",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename(columns={"A": "jumlah_aktif_a", "M": "jumlah_mbkm_m"})
)
for col in ["jumlah_aktif_a", "jumlah_mbkm_m"]:
    if col not in active_a.columns:
        active_a[col] = 0

# ---------- Agregasi sumber: mahasiswa baru ----------
raw_mahasiswa_baru["jumlah_mhs_baru"] = pd.to_numeric(
    raw_mahasiswa_baru["jumlah_mhs_baru"], errors="coerce"
)
raw_mahasiswa_baru = add_keys(
    raw_mahasiswa_baru,
    include_level=True,
    include_stem=True,
)

HIGH_ACCREDITATION = {"A", "UNGGUL", "TERAKREDITASI UNGGUL"}

raw_mahasiswa_baru["_akred_pt_key"] = raw_mahasiswa_baru["akred_pt"].map(
    normalize_category
)
raw_mahasiswa_baru["_akred_prodi_key"] = raw_mahasiswa_baru[
    "akred_prodi"
].map(normalize_category)

valid_new = raw_mahasiswa_baru.loc[
    raw_mahasiswa_baru["jumlah_mhs_baru"].notna()
].copy()
valid_new["_pt_high_value"] = np.where(
    valid_new["_akred_pt_key"].isin(HIGH_ACCREDITATION),
    valid_new["jumlah_mhs_baru"],
    0,
)
valid_new["_prodi_high_value"] = np.where(
    valid_new["_akred_prodi_key"].isin(HIGH_ACCREDITATION),
    valid_new["jumlah_mhs_baru"],
    0,
)
valid_new["_pt_prodi_high_value"] = np.where(
    valid_new["_akred_pt_key"].isin(HIGH_ACCREDITATION)
    & valid_new["_akred_prodi_key"].isin(HIGH_ACCREDITATION),
    valid_new["jumlah_mhs_baru"],
    0,
)

mahasiswa_baru_b = (
    valid_new.groupby(KEY_B, dropna=False, as_index=False)
    .agg(
        jumlah_mahasiswa_baru=("jumlah_mhs_baru", "sum"),
        mhs_baru_pt_akreditasi_tinggi=("_pt_high_value", "sum"),
        mhs_baru_prodi_akreditasi_tinggi=("_prodi_high_value", "sum"),
        mhs_baru_pt_prodi_tinggi=("_pt_prodi_high_value", "sum"),
    )
)

# ---------- Agregasi sumber: perguruan tinggi ----------
raw_pt = add_keys(raw_pt)
raw_pt["_akred_key"] = raw_pt["akred_pt"].map(normalize_category)
pt_unique = raw_pt.drop_duplicates(KEY_GEO + ["npsn"]).copy()
pt_unique["_is_high"] = pt_unique["_akred_key"].isin(HIGH_ACCREDITATION)

pt_geo = (
    pt_unique.groupby(KEY_GEO, dropna=False, as_index=False)
    .agg(
        jumlah_pt=("npsn", "nunique"),
        jumlah_pt_akreditasi_tinggi=("_is_high", "sum"),
    )
)

# ---------- Agregasi sumber: dosen jabatan ----------
raw_dosen_jabatan["jumlah"] = pd.to_numeric(
    raw_dosen_jabatan["jumlah"], errors="coerce"
)
raw_dosen_jabatan = add_keys(raw_dosen_jabatan)
raw_dosen_jabatan["_jabatan_key"] = raw_dosen_jabatan[
    "jabatan_akademik"
].map(normalize_category)
raw_dosen_jabatan["_profesor_value"] = np.where(
    raw_dosen_jabatan["_jabatan_key"].eq("PROFESOR"),
    raw_dosen_jabatan["jumlah"],
    0,
)
raw_dosen_jabatan["_senior_value"] = np.where(
    raw_dosen_jabatan["_jabatan_key"].isin(["LEKTOR KEPALA", "PROFESOR"]),
    raw_dosen_jabatan["jumlah"],
    0,
)

dosen_jabatan_geo = (
    raw_dosen_jabatan.loc[raw_dosen_jabatan["jumlah"].notna()]
    .groupby(KEY_GEO, dropna=False, as_index=False)
    .agg(
        dosen_total_jabatan=("jumlah", "sum"),
        dosen_profesor=("_profesor_value", "sum"),
        dosen_senior=("_senior_value", "sum"),
    )
)

# ---------- Agregasi sumber: dosen sertifikasi ----------
raw_dosen_sertifikasi["jumlah"] = pd.to_numeric(
    raw_dosen_sertifikasi["jumlah"], errors="coerce"
)
raw_dosen_sertifikasi = add_keys(raw_dosen_sertifikasi)
raw_dosen_sertifikasi["_sertifikasi_key"] = raw_dosen_sertifikasi[
    "status_sertifikasi"
].map(normalize_category)
certified_mask = (
    raw_dosen_sertifikasi["_sertifikasi_key"].str.contains(
        "SERTIFIKASI", na=False
    )
    & ~raw_dosen_sertifikasi["_sertifikasi_key"].str.contains(
        "BELUM", na=False
    )
)
raw_dosen_sertifikasi["_certified_value"] = np.where(
    certified_mask,
    raw_dosen_sertifikasi["jumlah"],
    0,
)

dosen_sertifikasi_geo = (
    raw_dosen_sertifikasi.loc[raw_dosen_sertifikasi["jumlah"].notna()]
    .groupby(KEY_GEO, dropna=False, as_index=False)
    .agg(
        dosen_total_sertifikasi=("jumlah", "sum"),
        dosen_tersertifikasi=("_certified_value", "sum"),
    )
)

# ---------- Agregasi sumber: dosen doktor STEM/non-STEM ----------
raw_dosen_doktor["jumlah"] = pd.to_numeric(
    raw_dosen_doktor["jumlah"], errors="coerce"
)
raw_dosen_doktor = add_keys(raw_dosen_doktor)
raw_dosen_doktor["_bidang_key"] = raw_dosen_doktor["bidang"].map(
    normalize_category
).replace({"NON-STEM": "NON STEM", "NONSTEM": "NON STEM"})

doctor_long = (
    raw_dosen_doktor.loc[
        raw_dosen_doktor["jumlah"].notna()
        & raw_dosen_doktor["_bidang_key"].isin(["STEM", "NON STEM"])
    ]
    .groupby(KEY_GEO + ["_bidang_key"], dropna=False, as_index=False)["jumlah"]
    .sum()
)

doctor_wide = (
    doctor_long.pivot_table(
        index=KEY_GEO,
        columns="_bidang_key",
        values="jumlah",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
    .rename(
        columns={
            "STEM": "dosen_doktor_stem",
            "NON STEM": "dosen_doktor_non_stem",
        }
    )
)
for col in ["dosen_doktor_stem", "dosen_doktor_non_stem"]:
    if col not in doctor_wide.columns:
        doctor_wide[col] = 0
doctor_wide["dosen_doktor_total"] = (
    doctor_wide["dosen_doktor_stem"]
    + doctor_wide["dosen_doktor_non_stem"]
)

doctor_sesuai = doctor_long.rename(
    columns={
        "_bidang_key": "_stem_key",
        "jumlah": "dosen_doktor_sesuai_bidang",
    }
)

print("Seluruh sumber telah diagregasi ke kunci analisis yang sesuai.")


In [ ]:

# Menyiapkan tabel gabungan unik untuk proses rekonsiliasi.
merged_a = gender_df[
    KEY_A
    + [
        "jumlah_lulusan",
        "jumlah_terdaftar",
        "jumlah_aktif_a",
        "jumlah_mbkm_m",
    ]
].copy()

merged_b = stem_df[
    KEY_B
    + [
        "jumlah_lulusan",
        "jumlah_mahasiswa_baru",
        "mhs_baru_pt_akreditasi_tinggi",
        "mhs_baru_prodi_akreditasi_tinggi",
        "mhs_baru_pt_prodi_tinggi",
    ]
].copy()

geo_metrics = [
    "jumlah_pt",
    "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan",
    "dosen_profesor",
    "dosen_senior",
    "dosen_total_sertifikasi",
    "dosen_tersertifikasi",
]
merged_geo = collapse_repeated_metrics(
    gender_df,
    KEY_GEO,
    geo_metrics,
    "Indikator wilayah dalam AD_Gender_Jenjang",
)

doctor_geo_metrics = [
    "dosen_doktor_stem",
    "dosen_doktor_non_stem",
    "dosen_doktor_total",
]
merged_doctor_geo = collapse_repeated_metrics(
    stem_df,
    KEY_GEO,
    doctor_geo_metrics,
    "Indikator dosen doktor dalam AD_STEM_Jenjang",
)

merged_doctor_sesuai = collapse_repeated_metrics(
    stem_df,
    KEY_GEO + ["_stem_key"],
    ["dosen_doktor_sesuai_bidang"],
    "Dosen doktor sesuai bidang dalam AD_STEM_Jenjang",
)

reconciliation_reports = []
reconciliation_discrepancies = []

def run_reconciliation(source, merged, keys, pairs, label):
    report, discrepancy = reconcile_metrics(
        source,
        merged,
        keys=keys,
        metric_pairs=pairs,
        source_label=label,
    )
    reconciliation_reports.append(report)
    if not discrepancy.empty:
        reconciliation_discrepancies.append(discrepancy)

run_reconciliation(
    lulus_a,
    merged_a[KEY_A + ["jumlah_lulusan"]],
    KEY_A,
    [("jumlah_lulusan", "jumlah_lulusan")],
    "Mahasiswa Lulus -> Dataset A",
)
run_reconciliation(
    lulus_b,
    merged_b[KEY_B + ["jumlah_lulusan"]],
    KEY_B,
    [("jumlah_lulusan", "jumlah_lulusan")],
    "Mahasiswa Lulus -> Dataset B",
)
run_reconciliation(
    terdaftar_a,
    merged_a[KEY_A + ["jumlah_terdaftar"]],
    KEY_A,
    [("jumlah_terdaftar", "jumlah_terdaftar")],
    "Mahasiswa Terdaftar -> Dataset A",
)
run_reconciliation(
    active_a,
    merged_a[KEY_A + ["jumlah_aktif_a", "jumlah_mbkm_m"]],
    KEY_A,
    [
        ("jumlah_aktif_a", "jumlah_aktif_a"),
        ("jumlah_mbkm_m", "jumlah_mbkm_m"),
    ],
    "Mahasiswa Aktif -> Dataset A",
)
run_reconciliation(
    mahasiswa_baru_b,
    merged_b[
        KEY_B
        + [
            "jumlah_mahasiswa_baru",
            "mhs_baru_pt_akreditasi_tinggi",
            "mhs_baru_prodi_akreditasi_tinggi",
            "mhs_baru_pt_prodi_tinggi",
        ]
    ],
    KEY_B,
    [
        ("jumlah_mahasiswa_baru", "jumlah_mahasiswa_baru"),
        (
            "mhs_baru_pt_akreditasi_tinggi",
            "mhs_baru_pt_akreditasi_tinggi",
        ),
        (
            "mhs_baru_prodi_akreditasi_tinggi",
            "mhs_baru_prodi_akreditasi_tinggi",
        ),
        ("mhs_baru_pt_prodi_tinggi", "mhs_baru_pt_prodi_tinggi"),
    ],
    "Mahasiswa Baru -> Dataset B",
)
run_reconciliation(
    pt_geo,
    merged_geo[
        KEY_GEO + ["jumlah_pt", "jumlah_pt_akreditasi_tinggi"]
    ],
    KEY_GEO,
    [
        ("jumlah_pt", "jumlah_pt"),
        (
            "jumlah_pt_akreditasi_tinggi",
            "jumlah_pt_akreditasi_tinggi",
        ),
    ],
    "Perguruan Tinggi -> Wilayah",
)
run_reconciliation(
    dosen_jabatan_geo,
    merged_geo[
        KEY_GEO + ["dosen_total_jabatan", "dosen_profesor", "dosen_senior"]
    ],
    KEY_GEO,
    [
        ("dosen_total_jabatan", "dosen_total_jabatan"),
        ("dosen_profesor", "dosen_profesor"),
        ("dosen_senior", "dosen_senior"),
    ],
    "Dosen Jabatan Akademik -> Wilayah",
)
run_reconciliation(
    dosen_sertifikasi_geo,
    merged_geo[
        KEY_GEO + ["dosen_total_sertifikasi", "dosen_tersertifikasi"]
    ],
    KEY_GEO,
    [
        ("dosen_total_sertifikasi", "dosen_total_sertifikasi"),
        ("dosen_tersertifikasi", "dosen_tersertifikasi"),
    ],
    "Dosen Sertifikasi -> Wilayah",
)
run_reconciliation(
    doctor_wide,
    merged_doctor_geo,
    KEY_GEO,
    [
        ("dosen_doktor_stem", "dosen_doktor_stem"),
        ("dosen_doktor_non_stem", "dosen_doktor_non_stem"),
        ("dosen_doktor_total", "dosen_doktor_total"),
    ],
    "Dosen Doktor STEM/Non-STEM -> Wilayah",
)
run_reconciliation(
    doctor_sesuai,
    merged_doctor_sesuai,
    KEY_GEO + ["_stem_key"],
    [
        (
            "dosen_doktor_sesuai_bidang",
            "dosen_doktor_sesuai_bidang",
        )
    ],
    "Dosen Doktor Sesuai Bidang -> Dataset B",
)

reconciliation_report = pd.concat(
    reconciliation_reports,
    ignore_index=True,
)
reconciliation_discrepancy = (
    pd.concat(reconciliation_discrepancies, ignore_index=True)
    if reconciliation_discrepancies
    else pd.DataFrame()
)

reconciliation_report.to_csv(
    OUTPUT_TABLE_DIR / "00_source_reconciliation.csv",
    index=False,
)
reconciliation_discrepancy.to_csv(
    OUTPUT_TABLE_DIR / "00_reconciliation_discrepancies.csv",
    index=False,
)

display(reconciliation_report)

failed = reconciliation_report.loc[
    reconciliation_report["status"].ne("PASS")
]
assert failed.empty, (
    "Rekonsiliasi sumber gagal. Periksa "
    "outputs/tables/00_reconciliation_discrepancies.csv\n"
    + failed.to_string(index=False)
)

print("PASS — seluruh nilai pada kunci yang dapat dibandingkan identik.")



## 6. Penetapan desain penelitian

Desain dikunci sebelum analisis agar target, exposure, prediktor, dan skema validasi tidak berubah setelah melihat performa model.


In [ ]:

research_design = pd.DataFrame(
    [
        {
            "analytical_dataset": "AD_Gender_Jenjang",
            "unit_of_analysis": (
                "provinsi × kabupaten/kota × jenis kelamin × jenjang"
            ),
            "primary_target": "jumlah_lulusan",
            "exposure_offset": "jumlah_terdaftar",
            "categorical_predictors": "jk; nm_jenj_didik",
            "numeric_predictors": (
                "jumlah_aktif_a; jumlah_mbkm_m; proporsi_mbkm; "
                "jumlah_pt; proporsi_pt_akreditasi_tinggi; "
                "proporsi_profesor; proporsi_dosen_senior; "
                "proporsi_dosen_tersertifikasi"
            ),
            "primary_validation_group": "provinsi_pt",
            "secondary_cluster": "kabupaten_pt",
            "primary_model_family": (
                "Negative Binomial dengan log(jumlah_terdaftar) sebagai offset"
            ),
            "interpretation_boundary": (
                "Ekologis dan cross-sectional; bukan peluang individu untuk lulus"
            ),
        },
        {
            "analytical_dataset": "AD_STEM_Jenjang",
            "unit_of_analysis": (
                "provinsi × kabupaten/kota × STEM/NON STEM × jenjang"
            ),
            "primary_target": "jumlah_lulusan",
            "exposure_offset": "jumlah_mahasiswa_baru (analisis flow, bukan cohort)",
            "categorical_predictors": "a_stem; nm_jenj_didik",
            "numeric_predictors": (
                "mhs_baru_pt_akreditasi_tinggi; "
                "mhs_baru_prodi_akreditasi_tinggi; "
                "jumlah_pt; proporsi_pt_akreditasi_tinggi; "
                "dosen_doktor_sesuai_bidang; proporsi_doktor_stem; "
                "proporsi_profesor; proporsi_dosen_tersertifikasi"
            ),
            "primary_validation_group": "provinsi_pt",
            "secondary_cluster": "kabupaten_pt",
            "primary_model_family": (
                "Negative Binomial/Poisson-Tweedie dan GroupKFold provinsi"
            ),
            "interpretation_boundary": (
                "graduate-to-entry flow ratio bukan completion rate"
            ),
        },
    ]
)

research_design.to_csv(
    OUTPUT_TABLE_DIR / "00_research_design.csv",
    index=False,
)
display(research_design.T)



## 7. Dataset tervalidasi dan checklist kelayakan

Kolom kunci internal yang diawali `_` hanya digunakan untuk audit dan tidak disimpan sebagai prediktor penelitian.


In [ ]:

internal_key_cols_a = [c for c in gender_df.columns if c.startswith("_")]
internal_key_cols_b = [c for c in stem_df.columns if c.startswith("_")]

validated_gender = gender_df.drop(columns=internal_key_cols_a)
validated_stem = stem_df.drop(columns=internal_key_cols_b)

validated_gender.to_csv(
    OUTPUT_TABLE_DIR / "00_validated_AD_Gender_Jenjang.csv",
    index=False,
)
validated_stem.to_csv(
    OUTPUT_TABLE_DIR / "00_validated_AD_STEM_Jenjang.csv",
    index=False,
)

readiness = pd.DataFrame(
    [
        {
            "check": "Workbook dan sheet dapat dibaca",
            "status": "PASS",
            "evidence": (
                f"{len(available_sheets)} sheet tersedia; "
                f"Dataset A={len(gender_df):,}, Dataset B={len(stem_df):,}"
            ),
        },
        {
            "check": "Nama kolom konsisten",
            "status": "PASS",
            "evidence": (
                f"{len(EXPECTED_GENDER_COLUMNS)} kolom wajib A dan "
                f"{len(EXPECTED_STEM_COLUMNS)} kolom wajib B tersedia"
            ),
        },
        {
            "check": "Tidak ada kunci duplikat",
            "status": "PASS",
            "evidence": "Kunci Dataset A dan B seluruhnya unik",
        },
        {
            "check": "Total/agregasi sumber tidak berubah",
            "status": (
                "PASS"
                if reconciliation_report["status"].eq("PASS").all()
                else "FAIL"
            ),
            "evidence": (
                f"{len(reconciliation_report)} pemeriksaan metrik; "
                f"{int(reconciliation_report['mismatch_keys'].sum())} mismatch"
            ),
        },
        {
            "check": "Missing values terdokumentasi",
            "status": "PASS",
            "evidence": "00_missing_values.csv",
        },
        {
            "check": "Unmatched keys terdokumentasi",
            "status": "PASS",
            "evidence": (
                f"00_unmatched_keys.csv; {len(audit_unmatched):,} baris audit"
            ),
        },
        {
            "check": "Desain penelitian ditetapkan",
            "status": "PASS",
            "evidence": "00_research_design.csv",
        },
    ]
)

readiness.to_csv(
    OUTPUT_TABLE_DIR / "00_readiness_checklist.csv",
    index=False,
)

run_metadata = {
    "random_seed": RANDOM_SEED,
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "combined_file": str(COMBINED_FILE),
    "dataset_a_rows": len(gender_df),
    "dataset_b_rows": len(stem_df),
    "dataset_a_total_lulusan": float(gender_df["jumlah_lulusan"].sum()),
    "dataset_b_total_lulusan": float(stem_df["jumlah_lulusan"].sum()),
    "reconciliation_mismatch_keys": int(
        reconciliation_report["mismatch_keys"].sum()
    ),
}

with (OUTPUT_TABLE_DIR / "00_run_metadata.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(run_metadata, handle, ensure_ascii=False, indent=2)

display(readiness)

assert readiness["status"].eq("PASS").all(), (
    "Notebook 00 belum lolos seluruh pemeriksaan."
)

print("=" * 72)
print("NOTEBOOK 00 LOLOS.")
print("Dataset siap dilanjutkan ke EDA setelah output audit ditinjau.")
print("=" * 72)
